# Lesson 9 exercise: ship it

**Goal:** merge your LoRA adapter into the model, prove the merged model behaves the same, push it to the Hugging Face Hub with a model card, and load it back by name like a stranger would.

**Before you start (2 minutes, needed for the push)**
1. Create a free account at [huggingface.co/join](https://huggingface.co/join).
2. Create a token with the **Write** role at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens).
3. In Colab, click the **key icon** (Secrets) in the left sidebar → *Add new secret* → name `HF_TOKEN`, value = your token → switch on *Notebook access*.

No token? The notebook still runs. The push cells skip, and the last cell loads your model from disk instead.

**How to use this notebook**
1. In Colab: *Runtime → Change runtime type → T4 GPU*.
2. Run the cells top to bottom. Wherever you see `...` after a `# TODO`, replace it with your code.
3. Run the **check** cell under each TODO. ✅ means you've got it; ❌ gives you a hint.

Stuck? Look up the error in the course's troubleshooting sheet (`reference/troubleshooting.html`). Still stuck? Ask on the Hugging Face Forums with your code and the full traceback.

## 0 · Setup, task data and metric
The same three cells as Lessons 5–8.

In [ ]:
%pip install -q -U "transformers>=5.19,<6" "trl>=1.15,<2" "peft>=0.21,<1" "datasets>=5,<6" "accelerate>=1.15,<2"
import torch, transformers
device = "cuda" if torch.cuda.is_available() else "cpu"
print("transformers", transformers.__version__, "| torch", torch.__version__, "| device:", device)

def check(passed, ok_msg, hint):
    print(("✅ " + ok_msg) if passed else ("❌ " + hint))

In [ ]:
import json, random

PROMPT = "Extract the person's name, age and city as JSON."
NAMES = ["Sara", "Omar", "Lina", "Youssef", "Mona", "Karim", "Nour", "Emma", "Liam", "Sofia",
         "Mateo", "Aiko", "Kenji", "Priya", "Arjun", "Chloe", "Lucas", "Fatima", "Diego", "Elena",
         "Leo", "Amara", "Kofi", "Hana", "Ines"]          # last 5 are test-only
CITIES = ["Cairo", "Alexandria", "Paris", "Lagos", "Tokyo", "Lima", "Berlin", "Toronto",
          "Mumbai", "Seoul", "Nairobi", "Madrid", "Dubai", "Oslo", "Austin", "Lisbon",
          "Hanoi", "Accra", "Quito", "Perth"]                # last 4 are test-only
TEMPLATES = [
    "Hi, I'm {name} from {city}. I'm {age}.",
    "{name} ({age}) lives in {city}.",
    "My name is {name}, I am {age} years old and I live in {city}.",
    "New signup: {name}, age {age}, based in {city}.",
    "Hello from {city}! {name} here, just turned {age}.",
    "{city}-based {name}, {age}, is joining the team next week.",
    "Contact: {name} | {age} | {city}",
    "Applicant {name} is {age} and currently resides in {city}.",
    "Meet {name}: {age} years young and loving life in {city}.",   # test-only
    "I moved to {city} last year. Name's {name}, age {age}.",      # test-only
]

def make_rows(n, split="train", seed=0):
    """Return n chat examples in TRL's prompt-completion format."""
    rng = random.Random(seed)
    test = split == "test"
    names = NAMES[20:] if test else NAMES[:20]
    cities = CITIES[16:] if test else CITIES[:16]
    templates = TEMPLATES[8:] if test else TEMPLATES[:8]
    rows = []
    for _ in range(n):
        person = {"name": rng.choice(names), "age": rng.randint(18, 80), "city": rng.choice(cities)}
        text = rng.choice(templates).format(**person)
        rows.append({
            "prompt": [{"role": "user", "content": f"{PROMPT}\n\nText: {text}"}],
            "completion": [{"role": "assistant", "content": json.dumps(person)}],
        })
    return rows

In [ ]:
def score(model, tokenizer, rows, show=3):
    """Greedy-generate a reply for each row; report valid-JSON rate and field accuracy."""
    model.eval()
    valid, fields_right = 0, 0
    for i, row in enumerate(rows):
        gold = json.loads(row["completion"][0]["content"])
        inputs = tokenizer.apply_chat_template(row["prompt"], add_generation_prompt=True,
                                               return_tensors="pt", return_dict=True).to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=40, do_sample=False)
        reply = tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        try:
            pred = json.loads(reply)
            valid += 1
        except json.JSONDecodeError:
            pred = {}
        if isinstance(pred, dict):
            fields_right += sum(pred.get(k) == v for k, v in gold.items())
        if i < show:
            print(f"  reply: {reply!r}")
    n = len(rows)
    result = {"valid_json": valid / n, "field_accuracy": fields_right / (3 * n)}
    print(result)
    return result

## 1 · Retrain Lesson 6's adapter
Colab forgot everything when your last session ended, so this cell reruns Lesson 6: LoRA on `SmolLM2-360M-Instruct`, then saves the adapter. It takes a few minutes, mostly scoring. Read the lesson while it runs.

In [ ]:
import os, gc, time
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

model_id = "HuggingFaceTB/SmolLM2-360M-Instruct"
adapter_dir = "smollm2-360m-json-lora"
tok = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float32).to(device)
test_rows = make_rows(40, "test", seed=1)

print("BASE model:")
before = score(model, tok, test_rows, show=1)          # the "before" row of your model card

args = SFTConfig(output_dir="lora-out", num_train_epochs=1, per_device_train_batch_size=8,
                 learning_rate=2e-4, bf16=False, max_length=256,
                 logging_steps=10, save_strategy="no", report_to="none")
peft_config = LoraConfig(r=8, lora_alpha=16, target_modules="all-linear", task_type="CAUSAL_LM")
trainer = SFTTrainer(model=model, args=args, train_dataset=Dataset.from_list(make_rows(500, "train", seed=0)),
                     processing_class=tok, peft_config=peft_config)
trainer.train()

print("LoRA model (base + adapter):")
t0 = time.time()
lora_scores = score(trainer.model, tok, test_rows, show=1)
lora_seconds = time.time() - t0
trainer.model.save_pretrained(adapter_dir)              # adapter only, as in Lesson 6
print(sorted(os.listdir(adapter_dir)))

## 2 · Merge the adapter into the model
To merge, load a fresh copy of the base model, attach the saved adapter with `PeftModel.from_pretrained`, then call `merge_and_unload()`. That adds each `B·A` into the frozen weight it belongs to and returns a plain Transformers model, with no adapter left.

**TODO 1:** Attach the adapter in `adapter_dir` to `base`, then merge. One line is enough.

In [ ]:
from peft import PeftModel

trainer = model = None; gc.collect(); torch.cuda.empty_cache()   # free the training copy
base = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float32).to(device)

# TODO 1
merged = ...

print(type(merged).__name__)

In [ ]:
# check 1
no_lora = hasattr(merged, "named_modules") and not any("lora" in n for n, _ in merged.named_modules())
check(not isinstance(merged, PeftModel) and no_lora,
      "A plain Transformers model: the adapter now lives inside the weights.",
      "Two steps: attach the adapter to base with PEFT, then merge it away. The lesson's section 1 shows both.")

## 3 · Save it, reload it, score it again
Save the tokenizer next to the model: the chat template lives in the tokenizer, and without it nobody can format prompts the way you trained.

**TODO 2:** Save `merged` **and** `tok` into `out_dir`.

In [ ]:
out_dir = "smollm2-360m-json-extractor"

# TODO 2
...

In [ ]:
# check 2
files = set(os.listdir(out_dir)) if os.path.isdir(out_dir) else set()
check({"config.json", "model.safetensors"} <= files, "Model weights and config saved.",
      "Save the merged model into out_dir, the same way section 1 saved the adapter.")
check({"tokenizer.json", "tokenizer_config.json"} <= files, "Tokenizer saved, chat template included.",
      "The tokenizer saves into a folder the same way the model does.")

Now load the folder exactly as a user would, and score it on the same 40 held-out rows.

In [ ]:
merged = base = None; gc.collect(); torch.cuda.empty_cache()
shipped = AutoModelForCausalLM.from_pretrained(out_dir, dtype=torch.float32).to(device)

print("Merged model, reloaded from disk:")
t0 = time.time()
shipped_scores = score(shipped, tok, test_rows, show=1)
shipped_seconds = time.time() - t0

check(shipped_scores == lora_scores, "Same scores as base + adapter: merging and saving changed nothing.",
      "The scores differ. Rerun sections 2 and 3, and make sure you merged the adapter you just trained.")
mb = lambda path: os.path.getsize(path) / 1e6
print(f"adapter file: {mb(os.path.join(adapter_dir, 'adapter_model.safetensors')):,.0f} MB | "
      f"merged model file: {mb(os.path.join(out_dir, 'model.safetensors')):,.0f} MB")
print(f"scoring 40 rows: {lora_seconds:.0f} s with the adapter attached, {shipped_seconds:.0f} s merged")

## 4 · Log in and pick a repo name
`get_token()` finds your Colab secret `HF_TOKEN` automatically. `whoami()` asks the Hub for your username.

In [ ]:
from huggingface_hub import get_token, whoami

repo_id = None
if get_token():
    repo_id = f"{whoami()['name']}/{out_dir}"
    print("Will push to https://huggingface.co/" + repo_id)
else:
    print("No token found, so the push cells will skip. The top of this notebook shows how to add HF_TOKEN.")

## 5 · Write the model card
The model card is the repo's `README.md`. The YAML block at the top is metadata for the Hub: `base_model` links your model to its parent. The text below it is for people. The numbers come straight from your variables, so the card can't drift from what you measured.

**TODO 3:** Write one or two sentences of limitations: where does this model break? Use your Lesson 8 failure tags. The cell then wraps `card_text` in a `ModelCard` and saves it as `README.md`.

In [ ]:
from huggingface_hub import ModelCard

# TODO 3
limitations = ...

name = repo_id or f"your-username/{out_dir}"
card_text = f"""---
base_model: {model_id}
library_name: transformers
license: apache-2.0
pipeline_tag: text-generation
tags: [json, information-extraction, lora, sft]
---

# {name}

Pulls a person's `name`, `age` and `city` out of a short English message and replies with JSON only.
Fine-tuned from [{model_id}](https://huggingface.co/{model_id}) with LoRA (r=8, all linear layers), then merged.

## How to use

```python
from transformers import pipeline
extractor = pipeline("text-generation", model="{name}")
prompt = "{PROMPT}\\n\\nText: Kwame, 34, just moved to Kumasi."
print(extractor([dict(role="user", content=prompt)], max_new_tokens=40)[0]["generated_text"][-1]["content"])
```

## Training data
500 synthetic messages built from 8 templates, 20 first names and 16 cities.

## Evaluation
40 held-out messages with unseen names, cities and templates, greedy decoding.

| model | valid-JSON rate | field accuracy |
|---|---|---|
| base `{model_id}` | {before['valid_json']:.2f} | {before['field_accuracy']:.2f} |
| this model | {shipped_scores['valid_json']:.2f} | {shipped_scores['field_accuracy']:.2f} |

## Intended use
A learning project: short, single-person English messages like the training data.

## Limitations
{limitations}
"""

card = ModelCard(card_text)

card.save(os.path.join(out_dir, "README.md"))
print(card_text)

In [ ]:
# check 3
check(isinstance(limitations, str) and len(limitations.split()) >= 8,
      "Limitations written.", "Write at least one full sentence (a string) about where the model breaks.")
check(isinstance(card, ModelCard) and card.data.base_model == model_id,
      "Card built, and the Hub can read its base_model metadata.",
      "The metadata didn't parse. Keep the --- block unindented at the very top of card_text.")

## 6 · Push to the Hub
`push_to_hub` creates the repo if it doesn't exist yet and uploads the files. Push the model you just tested (`shipped`) and the tokenizer. The card goes last, because a model push writes its own `README.md`.

**TODO 4:** Push `shipped` and `tok` to `repo_id`.

In [ ]:
if repo_id:
    # TODO 4
    ...
    card.push_to_hub(repo_id)
    print("Live at https://huggingface.co/" + repo_id)
else:
    print("Skipped: no token.")

In [ ]:
# check 4
if repo_id:
    from huggingface_hub import HfApi
    api = HfApi()
    on_hub = set(api.list_repo_files(repo_id)) if api.repo_exists(repo_id) else set()
    check({"model.safetensors", "config.json", "tokenizer.json", "README.md"} <= on_hub,
          "Model, tokenizer and card are on the Hub.",
          "Something is missing on the Hub. Push the model and the tokenizer (TODO 4), then rerun the push cell.")
    check("README.md" in on_hub and ModelCard.load(repo_id).data.base_model == model_id, "Your card (not the automatic one) is the README.",
          "Your card is not the README yet. Rerun the push cell: the card has to go after the model.")
else:
    check(False, "", "Nothing pushed: no token. Add HF_TOKEN (see the top of the notebook), then rerun sections 4–6.")

## 7 · Load it like a stranger would
Three lines, using only the repo id. With no token, this loads the folder on disk instead, which proves the same files work.

In [ ]:
from transformers import pipeline
extractor = pipeline("text-generation", model=repo_id or out_dir)
out = extractor([{"role": "user", "content": f"{PROMPT}\n\nText: Kwame, 34, just moved to Kumasi."}], max_new_tokens=40)

print(out[0]["generated_text"][-1]["content"])

## Stretch (optional)
1. **Load the adapter the short way.** `peft.AutoPeftModelForCausalLM.from_pretrained(adapter_dir)` reads the base model id from `adapter_config.json`, fetches the base model and attaches the adapter. Score it and compare.
2. **Halve the download.** Save a copy in float16 (`shipped.to(torch.float16)`) and check that the scores hold.
3. **Open your repo page** on the Hub. Find the "Finetuned from" link that `base_model` created.